In [ ]:
import pandas as pd

# Read the dataset
df = pd.read_csv('/content/train-test.csv')

# Display basic information and shape
print(f"Dataset Shape: {df.shape}")
print(df.head())

Dataset Shape: (48000, 14)
     load_id        pickup      delivery  pickup_lat  pickup_lon  \
0  TR-000001      Richmond     Baltimore    38.09122   -76.78906   
1  TR-000002      Richmond  Philadelphia    38.09122   -76.78906   
2  TR-000003  Philadelphia     Green Bay    39.22317   -72.96710   
3  TR-000004      Hartford       Atlanta    39.55328   -72.18051   
4  TR-000005        Dallas     Nashville    31.83025   -94.38343   

   delivery_lat  delivery_lon  distance equipment   weight        date  \
0      38.16908     -72.74564     274.3   Dry Van  30658.0  2025-01-01   
1      39.22317     -72.96710     280.5    Reefer  17555.0  2025-01-01   
2      44.30296     -87.52871     967.8   Dry Van  31721.0  2025-01-01   
3      34.84933     -86.28940     965.4   Dry Van  32333.0  2025-01-01   
4      35.29479     -88.08915     541.9    Reefer  35183.0  2025-01-01   

   market_index  quote_signal  posted_rate  
0       0.95684       2.39595       645.41  
1       0.97623       2.43355

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   load_id       48000 non-null  object 
 1   pickup        48000 non-null  object 
 2   delivery      48000 non-null  object 
 3   pickup_lat    48000 non-null  float64
 4   pickup_lon    48000 non-null  float64
 5   delivery_lat  48000 non-null  float64
 6   delivery_lon  48000 non-null  float64
 7   distance      48000 non-null  float64
 8   equipment     48000 non-null  object 
 9   weight        47700 non-null  float64
 10  date          48000 non-null  object 
 11  market_index  47626 non-null  float64
 12  quote_signal  48000 non-null  float64
 13  posted_rate   48000 non-null  float64
dtypes: float64(9), object(5)
memory usage: 5.1+ MB


In [ ]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
pickup_lat,48000.0,35.647545,4.315285,28.35765,31.98691,35.29479,39.411040,44.30296
pickup_lon,48000.0,-90.928964,13.482431,-121.69849,-98.40059,-88.08915,-83.285060,-69.50000
delivery_lat,48000.0,35.641175,4.317199,28.35765,31.98691,35.29479,39.411040,44.30296
delivery_lon,48000.0,-90.857310,13.476589,-121.69849,-98.40059,-87.52871,-83.285060,-69.50000
distance,48000.0,1135.856654,728.564416,70.00000,550.40000,953.30000,1645.525000,3439.80000
weight,47700.0,31028.844004,9391.440620,-47500.00000,25800.00000,31436.50000,37018.000000,47500.00000
market_index,47626.0,1.083387,0.168091,0.67639,0.94967,1.05580,1.219590,1.46778
quote_signal,48000.0,2.062468,0.291391,0.69228,1.89103,2.05575,2.221685,3.61035
posted_rate,48000.0,2373.980682,1486.493245,57.22000,1251.55500,2030.76000,3330.750000,25533.00000


In [ ]:
# Calculate medians
weight_median = df['weight'].median()
market_index_median = df['market_index'].median()

# Impute missing values
df['weight'] = df['weight'].fillna(weight_median)
df['market_index'] = df['market_index'].fillna(market_index_median)

# Verify that there are no missing values remaining
print(df[['weight', 'market_index']].isnull().sum())

weight          0
market_index    0
dtype: int64


In [ ]:
# Identify all numeric columns in the dataframe
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
print(f"Numeric columns to check for outliers: {numeric_cols}")

initial_rows = len(df)
mask = pd.Series(True, index=df.index)

# Calculate IQR and filter outliers for each numeric column
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Keep track of rows within valid bounds for each column
    mask = mask & (df[col] >= lower_bound) & (df[col] <= upper_bound)

# Filter the DataFrame in-place/overwrite
df = df[mask].copy()
removed_rows = initial_rows - len(df)

print(f"\nOriginal dataset size: {initial_rows}")
print(f"Cleaned dataset size: {len(df)}")
print(f"Removed {removed_rows} outlier rows ({removed_rows / initial_rows * 100:.2f}% of the data).")

Numeric columns to check for outliers: ['pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'weight', 'market_index', 'quote_signal', 'posted_rate']

Original dataset size: 48000
Cleaned dataset size: 44472
Removed 3528 outlier rows (7.35% of the data).


In [ ]:
# Show updated dataframe structure and information
print("=== Dataset Information ===")
df.info()

print("\n=== Summary Statistics ===")
print(df.describe().T)

print("\n=== First 5 Rows of Cleaned Dataset ===")
print(df.head())

=== Dataset Information ===
<class 'pandas.core.frame.DataFrame'>
Index: 44472 entries, 0 to 47999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   load_id       44472 non-null  object 
 1   pickup        44472 non-null  object 
 2   delivery      44472 non-null  object 
 3   pickup_lat    44472 non-null  float64
 4   pickup_lon    44472 non-null  float64
 5   delivery_lat  44472 non-null  float64
 6   delivery_lon  44472 non-null  float64
 7   distance      44472 non-null  float64
 8   equipment     44472 non-null  object 
 9   weight        44472 non-null  float64
 10  date          44472 non-null  object 
 11  market_index  44472 non-null  float64
 12  quote_signal  44472 non-null  float64
 13  posted_rate   44472 non-null  float64
dtypes: float64(9), object(5)
memory usage: 5.1+ MB

=== Summary Statistics ===
                count          mean          std         min           25%  \
pickup_lat    44472.0 

In [ ]:
# Convert date column to datetime
df['date'] = pd.to_datetime(df['date'])

# Sort the dataset chronologically
df = df.sort_values(by='date').reset_index(drop=True)
df.head(20)

,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000108,Philadelphia,St. Louis,39.22317,-72.96710,38.38204,-89.88819,1085.4,Dry Van,15712.0,2025-01-01,0.95874,1.78353,1958.10
2,TR-000107,San Antonio,Fort Wayne,29.50969,-98.40059,41.31561,-85.36206,1266.0,Flatbed,20678.0,2025-01-01,0.98745,1.99624,2498.90
3,TR-000106,Bakersfield,Milwaukee,33.28832,-116.67935,39.62180,-88.48008,1901.5,Dry Van,31374.0,2025-01-01,0.93499,1.81931,3407.03
4,TR-000105,Oklahoma City,Cincinnati,31.88294,-98.58097,37.15706,-86.77499,929.3,Dry Van,30918.0,2025-01-01,0.96079,1.91336,1792.83
5,TR-000104,Baton Rouge,Chattanooga,30.50134,-92.65374,35.29792,-87.18651,523.3,Dry Van,17804.0,2025-01-01,0.95108,2.05740,1074.67
6,TR-000103,Oklahoma City,Baton Rouge,31.88294,-98.58097,30.50134,-92.65374,418.0,Dry Van,31909.0,2025-01-01,0.94122,2.33556,970.69
7,TR-000102,New York,Houston,40.00674,-73.79840,30.71157,-94.76550,1556.3,Flatbed,28505.0,2025-01-01,0.92364,1.93531,3007.13
8,TR-000101,Oklahoma City,Cincinnati,31.88294,-98.58097,37.15706,-86.77499,900.3,Reefer,42482.0,2025-01-01,0.93478,2.30920,2101.86
9,TR-000100,Indianapolis,New York,38.52567,-88.85948,40.00674,-73.79840,932.8,Dry Van,37005.0,2025-01-01,0.99201,1.90185,1792.75


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44472 entries, 0 to 44471
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   load_id       44472 non-null  object        
 1   pickup        44472 non-null  object        
 2   delivery      44472 non-null  object        
 3   pickup_lat    44472 non-null  float64       
 4   pickup_lon    44472 non-null  float64       
 5   delivery_lat  44472 non-null  float64       
 6   delivery_lon  44472 non-null  float64       
 7   distance      44472 non-null  float64       
 8   equipment     44472 non-null  object        
 9   weight        44472 non-null  float64       
 10  date          44472 non-null  datetime64[ns]
 11  market_index  44472 non-null  float64       
 12  quote_signal  44472 non-null  float64       
 13  posted_rate   44472 non-null  float64       
dtypes: datetime64[ns](1), float64(9), object(4)
memory usage: 4.8+ MB


In [ ]:
# Show the minimum, maximum, and unique count of dates
min_date = df['date'].min()
max_date = df['date'].max()
unique_dates = df['date'].nunique()

print(f"Earliest Date: {min_date}")
print(f"Latest Date: {max_date}")
print(f"Total Unique Dates: {unique_dates}")

# Display the top 10 rows showcasing the sorted dates
print("\nFirst 10 dates in the dataset:")
print(df['date'].head(10))

print("\nLast 10 dates in the dataset:")
print(df['date'].tail(10))

Earliest Date: 2025-01-01 00:00:00
Latest Date: 2025-10-31 00:00:00
Total Unique Dates: 304

First 10 dates in the dataset:
0   2025-01-01
1   2025-01-01
2   2025-01-01
3   2025-01-01
4   2025-01-01
5   2025-01-01
6   2025-01-01
7   2025-01-01
8   2025-01-01
9   2025-01-01
Name: date, dtype: datetime64[ns]

Last 10 dates in the dataset:
44462   2025-10-31
44463   2025-10-31
44464   2025-10-31
44465   2025-10-31
44466   2025-10-31
44467   2025-10-31
44468   2025-10-31
44469   2025-10-31
44470   2025-10-31
44471   2025-10-31
Name: date, dtype: datetime64[ns]


In [ ]:
# Create year-month column
df["year_month"] = df["date"].dt.to_period("M")

# Get all months in order
months = sorted(df["year_month"].unique())

print("Months:")
for month in months:
    print(month)

Months:
2025-01
2025-02
2025-03
2025-04
2025-05
2025-06
2025-07
2025-08
2025-09
2025-10


In [ ]:
monthly_counts = df.groupby("year_month").size()

print(monthly_counts)

year_month
2025-01    4671
2025-02    4098
2025-03    4637
2025-04    4522
2025-05    4486
2025-06    4163
2025-07    4551
2025-08    4607
2025-09    4265
2025-10    4472
Freq: M, dtype: int64


In [ ]:
for i in range(1, len(months)):

    train_months = months[:i]
    validation_month = months[i]

    train_data = df[
        df["year_month"].isin(train_months)
    ]

    validation_data = df[
        df["year_month"] == validation_month
    ]

    print("Train:", train_months)
    print("Validation:", validation_month)
    print("Train rows:", len(train_data))
    print("Validation rows:", len(validation_data))
    print("-" * 50)

Train: [Period('2025-01', 'M')]
Validation: 2025-02
Train rows: 4671
Validation rows: 4098
--------------------------------------------------
Train: [Period('2025-01', 'M'), Period('2025-02', 'M')]
Validation: 2025-03
Train rows: 8769
Validation rows: 4637
--------------------------------------------------
Train: [Period('2025-01', 'M'), Period('2025-02', 'M'), Period('2025-03', 'M')]
Validation: 2025-04
Train rows: 13406
Validation rows: 4522
--------------------------------------------------
Train: [Period('2025-01', 'M'), Period('2025-02', 'M'), Period('2025-03', 'M'), Period('2025-04', 'M')]
Validation: 2025-05
Train rows: 17928
Validation rows: 4486
--------------------------------------------------
Train: [Period('2025-01', 'M'), Period('2025-02', 'M'), Period('2025-03', 'M'), Period('2025-04', 'M'), Period('2025-05', 'M')]
Validation: 2025-06
Train rows: 22414
Validation rows: 4163
--------------------------------------------------
Train: [Period('2025-01', 'M'), Period('2025-02

In [ ]:
target = "posted_rate"

In [ ]:
# Features
features = [
    col for col in df.columns
    if col not in [target, "date", "year_month","load_id"]
]

print("Features:")
print(features)

Features:
['pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'market_index', 'quote_signal']


In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.5 MB/s eta 0:00:00


In [ ]:
# Categorical features
categorical_features = [
    "pickup",
    "delivery",
    "equipment"
]

# Convert categorical columns to string
for col in categorical_features:
    df[col] = df[col].fillna("Unknown").astype(str)

In [ ]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

results = []

for i in range(1, len(months)):

    # -------------------------
    # Train / Validation split
    # -------------------------
    train_months = months[:i]
    validation_month = months[i]

    train_data = df[
        df["year_month"].isin(train_months)
    ].copy()

    validation_data = df[
        df["year_month"] == validation_month
    ].copy()

    # -------------------------
    # X / y
    # -------------------------
    X_train = train_data[features]
    y_train = train_data[target]

    X_val = validation_data[features]
    y_val = validation_data[target]

    # -------------------------
    # Model
    # -------------------------
    model = CatBoostRegressor(
    iterations=1200,
    learning_rate=0.05,
    depth=7,
    l2_leaf_reg=15,
    random_strength=5.0,
    loss_function="RMSE",
    random_seed=42,
    verbose=False
)

    # -------------------------
    # Train
    # -------------------------
    model.fit(
        X_train,
        y_train,
        cat_features=categorical_features
    )

    # -------------------------
    # Prediction
    # -------------------------
    predictions = model.predict(X_val)

    # -------------------------
    # Metrics
    # -------------------------
    rmse = np.sqrt(
        mean_squared_error(y_val, predictions)
    )

    mae = mean_absolute_error(
        y_val,
        predictions
    )

    results.append({
        "train_until": train_months[-1],
        "validation_month": validation_month,
        "train_rows": len(train_data),
        "validation_rows": len(validation_data),
        "RMSE": rmse,
        "MAE": mae
    })

    print(
        f"Train until {train_months[-1]} "
        f"→ Validate {validation_month} | "
        f"RMSE: {rmse:.2f} | "
        f"MAE: {mae:.2f}"
    )

Train until 2025-01 → Validate 2025-02 | RMSE: 242.13 | MAE: 80.27
Train until 2025-02 → Validate 2025-03 | RMSE: 265.69 | MAE: 103.94
Train until 2025-03 → Validate 2025-04 | RMSE: 242.34 | MAE: 127.89
Train until 2025-04 → Validate 2025-05 | RMSE: 235.64 | MAE: 74.06
Train until 2025-05 → Validate 2025-06 | RMSE: 267.51 | MAE: 108.16
Train until 2025-06 → Validate 2025-07 | RMSE: 239.76 | MAE: 76.93
Train until 2025-07 → Validate 2025-08 | RMSE: 231.45 | MAE: 84.83
Train until 2025-08 → Validate 2025-09 | RMSE: 257.20 | MAE: 121.95
Train until 2025-09 → Validate 2025-10 | RMSE: 259.77 | MAE: 103.26


In [ ]:
results_df = pd.DataFrame(results)

print(results_df)

  train_until validation_month  train_rows  validation_rows        RMSE  \
0     2025-01          2025-02        4671             4098  242.127866   
1     2025-02          2025-03        8769             4637  265.685584   
2     2025-03          2025-04       13406             4522  242.337511   
3     2025-04          2025-05       17928             4486  235.638966   
4     2025-05          2025-06       22414             4163  267.507142   
5     2025-06          2025-07       26577             4551  239.756366   
6     2025-07          2025-08       31128             4607  231.448224   
7     2025-08          2025-09       35735             4265  257.196773   
8     2025-09          2025-10       40000             4472  259.774426   

          MAE  
0   80.272395  
1  103.937997  
2  127.893537  
3   74.055121  
4  108.161225  
5   76.931274  
6   84.832441  
7  121.950298  
8  103.257557  


In [ ]:
# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Average metrics
average_rmse = results_df["RMSE"].mean()
average_mae = results_df["MAE"].mean()

print("\n===== EXPANDING WINDOW VALIDATION RESULTS =====")
print(results_df.to_string(index=False))

print("\n===== AVERAGE METRICS =====")
print(f"Average Monthly RMSE: {average_rmse:.2f}")
print(f"Average Monthly MAE:  {average_mae:.2f}")


===== EXPANDING WINDOW VALIDATION RESULTS =====
train_until validation_month  train_rows  validation_rows       RMSE        MAE
    2025-01          2025-02        4671             4098 242.127866  80.272395
    2025-02          2025-03        8769             4637 265.685584 103.937997
    2025-03          2025-04       13406             4522 242.337511 127.893537
    2025-04          2025-05       17928             4486 235.638966  74.055121
    2025-05          2025-06       22414             4163 267.507142 108.161225
    2025-06          2025-07       26577             4551 239.756366  76.931274
    2025-07          2025-08       31128             4607 231.448224  84.832441
    2025-08          2025-09       35735             4265 257.196773 121.950298
    2025-09          2025-10       40000             4472 259.774426 103.257557

===== AVERAGE METRICS =====
Average Monthly RMSE: 249.05
Average Monthly MAE:  97.92


In [ ]:
df_validation=pd.read_csv("/content/validation.csv")
df_validation.head(2)

,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal
0,TE-000001,Baton Rouge,Mobile,30.50134,-92.65374,31.80442,-88.25195,331.4,Flatbed,19958.0,2025-11-01,0.90402,1.86388
1,TE-000002,Savannah,Los Angeles,33.60973,-82.29994,28.56624,-116.70249,2406.2,Reefer,34114.0,2025-11-01,0.92851,1.86240


In [ ]:

import pandas as pd
import numpy as np

# 1. Load validation data
validation = pd.read_csv("/content/validation.csv")

# Keep every load_id
validation_ids = validation["load_id"].copy()

# 2. Convert date
validation["date"] = pd.to_datetime(
    validation["date"], errors="coerce"
)

# 3. Fill missing values using training medians
for col in ["weight", "market_index"]:
    if col in validation.columns:
        validation[col] = validation[col].fillna(df[col].median())

# 4. Handle categorical features
for col in categorical_features:
    if col in validation.columns:
        validation[col] = (
            validation[col]
            .fillna("Unknown")
            .astype(str)
        )

# 5. Select exactly the same features used for training
X_validation = validation.reindex(columns=features).copy()

# 6. Fill any remaining missing numeric values
for col in X_validation.columns:
    if col not in categorical_features:
        X_validation[col] = X_validation[col].fillna(
            df[col].median()
        )

# 7. Handle extreme numeric feature values
# Use training-data limits; do not remove validation rows
numeric_features = [
    col for col in features
    if col not in categorical_features
    and pd.api.types.is_numeric_dtype(df[col])
]

for col in numeric_features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    X_validation[col] = X_validation[col].clip(
        lower=lower, upper=upper
    )

# 8. Predict using your already-trained model
predictions = model.predict(X_validation)

template = pd.read_csv(
    "/content/validation-predictions-template.csv"
)

# تأكد من مطابقة ترتيب load_id بين الملفين قبل الإسناد
template["predicted_rate"] = predictions

template.to_csv("/content/validation-predictions-template.csv", index=False)

=== Preview of /content/validation.csv ===
Shape: (12000, 13)
Columns: ['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal']


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal
0,TE-000001,Baton Rouge,Mobile,30.50134,-92.65374,31.80442,-88.25195,331.4,Flatbed,19958.0,2025-11-01,0.90402,1.86388
1,TE-000002,Savannah,Los Angeles,33.60973,-82.29994,28.56624,-116.70249,2406.2,Reefer,34114.0,2025-11-01,0.92851,1.86240
2,TE-000003,San Francisco,Raleigh,35.19670,-121.69849,35.37376,-77.47605,2846.6,Dry Van,33435.0,2025-11-01,0.88117,1.95480
3,TE-000004,Dayton,Los Angeles,39.87206,-85.62931,28.56624,-116.70249,2261.6,Dry Van,25392.0,2025-11-01,0.89156,2.14579
4,TE-000005,Memphis,San Antonio,35.56802,-89.52871,29.50969,-98.40059,800.4,Flatbed,NaN,2025-11-01,0.88308,2.22541




=== Preview of /content/validation-predictions-template.csv ===
Shape: (12000, 2)
Columns: ['load_id', 'predicted_rate']


,load_id,predicted_rate
0,TE-000001,852.503259
1,TE-000002,4832.625077
2,TE-000003,4995.964327
3,TE-000004,4076.664134
4,TE-000005,1812.328466




=== Preview of /content/december-chart-inputs.csv ===
Shape: (31, 7)
Columns: ['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date', 'predicted_rate']


,pickup,delivery,distance,equipment,weight,date,predicted_rate
0,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-01,NaN
1,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-02,NaN
2,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-03,NaN
3,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-04,NaN
4,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-05,NaN


try catboost model after try feature

---

engineering

In [ ]:

import numpy as np
import pandas as pd

# Work on a copy
df = df.copy()

# --------------------------------------------------
# 1. Parse date
# --------------------------------------------------
df["date"] = pd.to_datetime(df["date"], errors="coerce")

df["year_month"] = df["date"].dt.to_period("M").astype(str)

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter
df["day_of_week"] = df["date"].dt.dayofweek
df["day_of_month"] = df["date"].dt.day
df["week_of_year"] = (
    df["date"].dt.isocalendar().week.astype("float")
)

# Cyclical calendar features: December is close to January
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

df["weekday_sin"] = np.sin(
    2 * np.pi * df["day_of_week"] / 7
)
df["weekday_cos"] = np.cos(
    2 * np.pi * df["day_of_week"] / 7
)

# --------------------------------------------------
# 2. Clean numeric feature representations
# --------------------------------------------------
numeric_columns = [
    "distance",
    "weight",
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon",
    "market_index",
    "quote_signal",
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# --------------------------------------------------
# 3. Distance features
# --------------------------------------------------
if "distance" in df.columns:
    # Log transformation reduces the effect of very large values
    df["log_distance"] = np.log1p(
        df["distance"].clip(lower=0)
    )

    # Fixed, interpretable distance bands
    df["distance_band"] = pd.cut(
        df["distance"],
        bins=[-np.inf, 100, 250, 500, 1000, 2000, np.inf],
        labels=[
            "Very_short",
            "Short",
            "Medium",
            "Long",
            "Very_long",
            "Extra_long",
        ],
    ).astype("object")

    # Avoid division by zero
    df["distance_safe"] = df["distance"].where(
        df["distance"] > 0
    )

# --------------------------------------------------
# 4. Weight features
# --------------------------------------------------
if "weight" in df.columns:
    # Nonpositive weights are treated as missing in derived features.
    # This does not delete the original records.
    df["weight_positive"] = df["weight"].where(
        df["weight"] > 0
    )

    df["log_weight"] = np.log1p(
        df["weight_positive"]
    )

    df["weight_band"] = pd.cut(
        df["weight_positive"],
        bins=[-np.inf, 5000, 10000, 20000, 40000, np.inf],
        labels=[
            "Light",
            "Medium",
            "Heavy",
            "Very_heavy",
            "Extra_heavy",
        ],
    ).astype("object")

    # Weight relative to shipment distance
    if "distance_safe" in df.columns:
        df["weight_per_mile"] = (
            df["weight_positive"] / df["distance_safe"]
        )

        df["log_weight_per_mile"] = np.log1p(
            df["weight_per_mile"].clip(lower=0)
        )

# --------------------------------------------------
# 5. Geographic features
# --------------------------------------------------
coordinate_columns = [
    "pickup_lat", "pickup_lon",
    "delivery_lat", "delivery_lon",
]

if all(col in df.columns for col in coordinate_columns):
    df["latitude_difference"] = (
        df["delivery_lat"] - df["pickup_lat"]
    )

    df["longitude_difference"] = (
        df["delivery_lon"] - df["pickup_lon"]
    )

    df["abs_latitude_difference"] = (
        df["latitude_difference"].abs()
    )

    df["abs_longitude_difference"] = (
        df["longitude_difference"].abs()
    )

    # Straight-line distance using the Haversine formula
    lat1 = np.radians(df["pickup_lat"])
    lon1 = np.radians(df["pickup_lon"])
    lat2 = np.radians(df["delivery_lat"])
    lon2 = np.radians(df["delivery_lon"])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    a = a.clip(0, 1)

    df["straight_line_distance"] = (
        2 * 3958.8
        * np.arcsin(np.sqrt(a))
    )

    # Compare road distance with straight-line distance
    if "distance" in df.columns:
        df["road_to_straight_ratio"] = (
            df["distance"]
            / df["straight_line_distance"].where(
                df["straight_line_distance"] > 0
            )
        )

# --------------------------------------------------
# 6. Route features
# --------------------------------------------------
if "pickup" in df.columns and "delivery" in df.columns:
    df["pickup"] = df["pickup"].fillna("Unknown").astype(str)
    df["delivery"] = df["delivery"].fillna("Unknown").astype(str)

    # Direction matters: A -> B is different from B -> A
    df["route"] = df["pickup"] + " -> " + df["delivery"]

# --------------------------------------------------
# 7. Market and equipment interactions
# --------------------------------------------------
if "equipment" in df.columns:
    df["equipment"] = (
        df["equipment"].fillna("Unknown").astype(str)
    )

    if "distance" in df.columns:
        df["equipment_distance"] = (
            df["equipment"] + "_" + df["distance_band"].astype(str)
            if "distance_band" in df.columns
            else df["equipment"]
        )

if "market_index" in df.columns and "distance" in df.columns:
    df["market_index_x_distance"] = (
        df["market_index"] * df["distance"]
    )

if "quote_signal" in df.columns and "distance" in df.columns:
    df["quote_signal_x_distance"] = (
        df["quote_signal"] * df["distance"]
    )

# --------------------------------------------------
# 8. Inspect the engineered dataset
# --------------------------------------------------
print("Dataset shape:", df.shape)
print("\nNew feature columns:")
print([
    col for col in df.columns
    if col not in numeric_columns
])

print("\nMissing values in engineered features:")
print(df.isnull().sum().sort_values(ascending=False).head(15))

Dataset shape: (44472, 43)

New feature columns:
['load_id', 'pickup', 'delivery', 'equipment', 'date', 'posted_rate', 'year_month', 'year', 'month', 'quarter', 'day_of_week', 'day_of_month', 'week_of_year', 'month_sin', 'month_cos', 'weekday_sin', 'weekday_cos', 'log_distance', 'distance_band', 'distance_safe', 'weight_positive', 'log_weight', 'weight_band', 'weight_per_mile', 'log_weight_per_mile', 'latitude_difference', 'longitude_difference', 'abs_latitude_difference', 'abs_longitude_difference', 'straight_line_distance', 'road_to_straight_ratio', 'route', 'equipment_distance', 'market_index_x_distance', 'quote_signal_x_distance']

Missing values in engineered features:
load_id         0
pickup          0
delivery        0
pickup_lat      0
pickup_lon      0
delivery_lat    0
delivery_lon    0
distance        0
equipment       0
weight          0
date            0
market_index    0
quote_signal    0
posted_rate     0
year_month      0
dtype: int64


In [ ]:

target = "posted_rate"

# Columns that should never be model inputs
excluded_columns = [
    target,
    "load_id",
    "date",
    "year_month",
]

features = [
    col for col in df.columns
    if col not in excluded_columns
]

categorical_features = [
    col for col in [
        "pickup",
        "delivery",
        "equipment",
        "route",
        "distance_band",
        "weight_band",
        "equipment_distance",
    ]
    if col in features
]

# CatBoost accepts categorical values as strings.
for col in categorical_features:
    df[col] = df[col].fillna("Unknown").astype(str)

print("Number of features:", len(features))
print("Categorical features:", categorical_features)
print("Features:", features)

Number of features: 39
Categorical features: ['pickup', 'delivery', 'equipment', 'route', 'distance_band', 'weight_band', 'equipment_distance']
Features: ['pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'market_index', 'quote_signal', 'year', 'month', 'quarter', 'day_of_week', 'day_of_month', 'week_of_year', 'month_sin', 'month_cos', 'weekday_sin', 'weekday_cos', 'log_distance', 'distance_band', 'distance_safe', 'weight_positive', 'log_weight', 'weight_band', 'weight_per_mile', 'log_weight_per_mile', 'latitude_difference', 'longitude_difference', 'abs_latitude_difference', 'abs_longitude_difference', 'straight_line_distance', 'road_to_straight_ratio', 'route', 'equipment_distance', 'market_index_x_distance', 'quote_signal_x_distance']


In [ ]:

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

# Do not train on rows without a target
model_df = df.dropna(subset=[target]).copy()

months = sorted(model_df["year_month"].dropna().unique())
results = []

for i in range(1, len(months)):
    train_months = months[:i]
    validation_month = months[i]

    train_data = model_df[
        model_df["year_month"].isin(train_months)
    ].copy()

    validation_data = model_df[
        model_df["year_month"] == validation_month
    ].copy()

    X_train = train_data[features].copy()
    y_train = train_data[target]

    X_val = validation_data[features].copy()
    y_val = validation_data[target]

    # Fill missing numerical values using training medians only.
    numeric_features = [
        col for col in features
        if col not in categorical_features
    ]

    medians = X_train[numeric_features].median()

    X_train[numeric_features] = (
        X_train[numeric_features].fillna(medians)
    )
    X_val[numeric_features] = (
        X_val[numeric_features].fillna(medians)
    )

    # Ensure categorical missing values are handled consistently
    for col in categorical_features:
        X_train[col] = X_train[col].fillna("Unknown").astype(str)
        X_val[col] = X_val[col].fillna("Unknown").astype(str)

    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=5,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=categorical_features,
    )

    predictions = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, predictions))
    mae = mean_absolute_error(y_val, predictions)

    results.append({
        "train_until": train_months[-1],
        "validation_month": validation_month,
        "train_rows": len(train_data),
        "validation_rows": len(validation_data),
        "RMSE": rmse,
        "MAE": mae,
    })

    print(
        f"Train until {train_months[-1]} -> "
        f"Validate {validation_month} | "
        f"RMSE: {rmse:.2f} | MAE: {mae:.2f}"
    )

results_df = pd.DataFrame(results)

if not results_df.empty:
    print("\nAverage monthly RMSE:", round(results_df["RMSE"].mean(), 2))
    print("Average monthly MAE:", round(results_df["MAE"].mean(), 2))
    display(results_df)

NameError: name 'features' is not defined

hyperparamter tunning

In [ ]:

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# =========================================
# 1. Prepare data
# =========================================
target = "posted_rate"

tuning_df = df.copy()

tuning_df["date"] = pd.to_datetime(
    tuning_df["date"], errors="coerce"
)
tuning_df["year_month"] = (
    tuning_df["date"].dt.to_period("M").astype(str)
)

tuning_df = tuning_df.dropna(
    subset=[target, "year_month"]
).copy()

excluded_columns = {
    target, "load_id", "date", "year_month"
}

features = [
    col for col in tuning_df.columns
    if col not in excluded_columns
]

# Detect categorical features
categorical_features = [
    col for col in features
    if (
        pd.api.types.is_object_dtype(tuning_df[col])
        or isinstance(tuning_df[col].dtype, pd.CategoricalDtype)
        or pd.api.types.is_string_dtype(tuning_df[col])
    )
]

numeric_features = [
    col for col in features
    if col not in categorical_features
]

for col in categorical_features:
    tuning_df[col] = (
        tuning_df[col].fillna("Unknown").astype(str)
    )

months = sorted(tuning_df["year_month"].unique())

if len(months) < 2:
    raise ValueError("At least two months are required.")

print("Months:", months)
print("Number of features:", len(features))
print("Categorical features:", categorical_features)

# =========================================
# 2. Define hyperparameter search space
# =========================================
param_distributions = {
    "depth": [4, 5, 6, 7, 8],
    "learning_rate": [0.02, 0.03, 0.05, 0.07],
    "l2_leaf_reg": [3, 5, 7, 10, 15],
    "random_strength": [0.1, 0.5, 1.0, 2.0, 5.0],
}

parameter_candidates = list(
    ParameterSampler(
        param_distributions,
        n_iter=3,
        random_state=42,
    )
)

# =========================================
# 3. Monthly expanding-window tuning
# =========================================
all_results = []

for trial, params in enumerate(parameter_candidates, start=1):

    monthly_rmses = []
    monthly_maes = []

    print(f"\nTrial {trial}/ {len(parameter_candidates)}")
    print("Parameters:", params)

    for i in range(1, len(months)):
        train_months = months[:i]
        validation_month = months[i]

        train_data = tuning_df[
            tuning_df["year_month"].isin(train_months)
        ]

        validation_data = tuning_df[
            tuning_df["year_month"] == validation_month
        ]

        X_train = train_data[features].copy()
        y_train = train_data[target]

        X_val = validation_data[features].copy()
        y_val = validation_data[target]

        # Fit imputation using training data only
        X_train[numeric_features] = (
            X_train[numeric_features]
            .replace([np.inf, -np.inf], np.nan)
        )

        X_val[numeric_features] = (
            X_val[numeric_features]
            .replace([np.inf, -np.inf], np.nan)
        )

        medians = X_train[numeric_features].median()
        medians = medians.fillna(0)

        X_train[numeric_features] = (
            X_train[numeric_features]
            .fillna(medians)
        )

        X_val[numeric_features] = (
            X_val[numeric_features]
            .fillna(medians)
        )

        for col in categorical_features:
            X_train[col] = (
                X_train[col].fillna("Unknown").astype(str)
            )
            X_val[col] = (
                X_val[col].fillna("Unknown").astype(str)
            )

        model = CatBoostRegressor(
            iterations=800,
            loss_function="RMSE",
            eval_metric="RMSE",
            random_seed=42,
            verbose=False,
            allow_writing_files=False,
            **params,
        )

        model.fit(
            X_train,
            y_train,
            cat_features=categorical_features,
            eval_set=(X_val, y_val),
            early_stopping_rounds=100,
            verbose=False,
        )

        predictions = model.predict(X_val)

        rmse = np.sqrt(
            mean_squared_error(y_val, predictions)
        )
        mae = mean_absolute_error(y_val, predictions)

        monthly_rmses.append(rmse)
        monthly_maes.append(mae)

        print(
            f"  Validate {validation_month}: "
            f"RMSE={rmse:.2f}, MAE={mae:.2f}"
        )

    all_results.append({
        "trial": trial,
        **params,
        "average_monthly_RMSE": np.mean(monthly_rmses),
        "average_monthly_MAE": np.mean(monthly_maes),
    })

    print(
        f"Trial average RMSE: {np.mean(monthly_rmses):.2f}"
    )

# =========================================
# 4. Compare all trials
# =========================================
tuning_results = pd.DataFrame(all_results).sort_values(
    "average_monthly_RMSE"
).reset_index(drop=True)

print("\n===== TUNING RESULTS =====")
print(tuning_results.to_string(index=False))

best_row = tuning_results.iloc[0]

best_params = {
    "depth": int(best_row["depth"]),
    "learning_rate": float(best_row["learning_rate"]),
    "l2_leaf_reg": float(best_row["l2_leaf_reg"]),
    "random_strength": float(best_row["random_strength"]),
}

print("\nBest parameters:", best_params)
print(
    "Best average monthly RMSE:",
    round(best_row["average_monthly_RMSE"], 2)
)
print(
    "Best average monthly MAE:",
    round(best_row["average_monthly_MAE"], 2)
)

Months: ['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10']
Number of features: 39
Categorical features: ['pickup', 'delivery', 'equipment', 'distance_band', 'weight_band', 'route', 'equipment_distance']

Trial 1/ 8
Parameters: {'random_strength': 0.5, 'learning_rate': 0.02, 'l2_leaf_reg': 10, 'depth': 7}
  Validate 2025-02: RMSE=235.96, MAE=70.60
  Validate 2025-03: RMSE=255.93, MAE=88.40
  Validate 2025-04: RMSE=293.21, MAE=183.65
  Validate 2025-05: RMSE=241.27, MAE=80.82
  Validate 2025-06: RMSE=265.89, MAE=107.26
  Validate 2025-07: RMSE=241.76, MAE=81.87
  Validate 2025-08: RMSE=223.95, MAE=68.55
  Validate 2025-09: RMSE=246.74, MAE=100.04
  Validate 2025-10: RMSE=253.61, MAE=80.10
Trial average RMSE: 250.92

Trial 2/ 8
Parameters: {'random_strength': 2.0, 'learning_rate': 0.05, 'l2_leaf_reg': 10, 'depth': 4}
  Validate 2025-02: RMSE=238.17, MAE=76.44
  Validate 2025-03: RMSE=256.62, MAE=90.82
  Validate 2025-04: RMSE=272